# 데이터 가져오기

## 독립변수

In [115]:
import requests
import pandas as pd
import urllib3

# SSL 경고 메시지 숨기기
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# API 키 입력
API_KEY = "5STQAATD94Z76TI7W683"

# ECOS open API로 데이터 가져오기
def get_ecos_data(stat_code, cycle, start_date, end_date, item_code, col_name):
    url = (
        f"https://ecos.bok.or.kr/api/StatisticSearch/"
        f"{API_KEY}/json/kr/1/3000/"                                   # 1~3000번째 데이터: 주말, 공휴일 제외 10년치 약 3000개
        f"{stat_code}/{cycle}/{start_date}/{end_date}/{item_code}"     # 통계표 코드/주기(D:일별, M:월별)/시작일/종료일/항목 코드
    )
    # 요청 보내기
    response = requests.get(url, verify=False)
    data = response.json()

    # 데이터 파싱
    rows = data["StatisticSearch"]["row"]
    df = pd.DataFrame(rows)

    # 필요한 컬럼만 정리
    df = df[["TIME", "DATA_VALUE"]].rename(columns={
        "TIME": "날짜",
        "DATA_VALUE": col_name
    })

    df[col_name] = pd.to_numeric(df[col_name])  # 숫자 타입으로 변환

    return df

1. 원/달러 환율, 한국&미국 기준금리

In [116]:
ex_rate = get_ecos_data("731Y001", "D", "20150101", "20241231", "0000001", "원/달러환율")
kr_rate = get_ecos_data("722Y001", "M", "201501", "202412", "0101000", "한국금리")
us_rate = get_ecos_data("902Y006", "M", "201501", "202412", "US", "미국금리")

print(ex_rate.shape)
print(ex_rate.head(10))

(2468, 2)
         날짜  원/달러환율
0  20150102  1099.2
1  20150105  1102.6
2  20150106  1108.7
3  20150107  1103.9
4  20150108  1099.7
5  20150109  1098.5
6  20150112  1092.2
7  20150113  1083.0
8  20150114  1081.5
9  20150115  1079.9


In [117]:
print(kr_rate.shape)
print(kr_rate.head(10))

(120, 2)
       날짜  한국금리
0  201501  2.00
1  201502  2.00
2  201503  1.75
3  201504  1.75
4  201505  1.75
5  201506  1.50
6  201507  1.50
7  201508  1.50
8  201509  1.50
9  201510  1.50


In [118]:
print(us_rate.shape)
print(us_rate.head(10))

(120, 2)
       날짜   미국금리
0  201501  0.125
1  201502  0.125
2  201503  0.125
3  201504  0.125
4  201505  0.125
5  201506  0.125
6  201507  0.125
7  201508  0.125
8  201509  0.125
9  201510  0.125


2. 소비자물가지수(CPI)

In [119]:
cpi = get_ecos_data("901Y009", "M", "201501", "202412", "0", "CPI")

print(cpi.shape)
print(cpi.head(10))

(120, 2)
       날짜     CPI
0  201501  94.643
1  201502  94.587
2  201503  94.596
3  201504  94.625
4  201505  94.890
5  201506  94.909
6  201507  95.080
7  201508  95.213
8  201509  94.966
9  201510  94.966


3. 유가

In [120]:
oil_price = pd.read_csv("국제_원유가격20150101_20241231.csv", encoding="euc-kr")

oil_price = oil_price.rename(columns={"기간": "날짜"})

wti   = oil_price[["날짜", "WTI"]].copy()
dubai = oil_price[["날짜", "Dubai"]].copy()

print(wti.head(3))
print(dubai.head(3))

          날짜    WTI
0  15년01월02일  52.69
1  15년01월05일  50.04
2  15년01월06일  47.93
          날짜  Dubai
0  15년01월02일  53.27
1  15년01월05일  50.98
2  15년01월06일  48.08


## 종속변수

In [49]:
%pwd

'C:\\Users\\926174\\Downloads'

1. 품목별 국가별 수출입 실적

In [63]:
import glob

# 폴더 경로 지정
folder = r"C:\Users\926174\Downloads\수출입 실적"

files = glob.glob(folder + r"\수출입 실적(품목별+국가별)_*.csv")

trade_list = []
for file in sorted(files):
    df = pd.read_csv(file, encoding='utf-8-sig')
    trade_list.append(df)
    print(f"{file} 로드 완료 → {df.shape}")

# 하나로 합치기
trade = pd.concat(trade_list, ignore_index=True)

# 총계 행 제거
trade = trade[trade['기간'] != '총계'].reset_index(drop=True)

print(f"\n최종 shape: {trade.shape}")
trade.head(5)

C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2015.csv 로드 완료 → (95804, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2016.csv 로드 완료 → (96814, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2017.csv 로드 완료 → (97724, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2018.csv 로드 완료 → (99286, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_201901_11.csv 로드 완료 → (92203, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_201912.csv 로드 완료 → (8458, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2020.csv 로드 완료 → (99617, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_202101_11.csv 로드 완료 → (93486, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_202112.csv 로드 완료 → (8593, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_202201_11.csv 로드 완료 → (92691, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_202212.csv 로드 완료 → (8477, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_202301_11.csv 로드 완료 → (93452, 9)
C:\Users\926174\Downloads\수출입 실적\수출입 실적(품목별+국가별)_2023

,기간,국가,HS코드,품목명,수출 중량,수출 금액,수입 중량,수입 금액,무역수지
0,2015-01,가나,3.0,어류ㆍ갑각류ㆍ연체동물과 그 밖의 수생(水生) 무척추동물,81.0,120,0.0,0,120
1,2015-01,가나,12.0,"채유(採油)에 적합한 종자와 과실, 각종 종자와 과실, 공업용ㆍ의약용 식물, 짚과 ...",0.0,2,0.0,0,2
2,2015-01,가나,15.0,"동물성ㆍ식물성ㆍ미생물성 지방과 기름 및이들의 분해생산물, 조제한 식용 지방과 동물성...",0.0,0,17.5,56,-56
3,2015-01,가나,17.0,당류(糖類)와 설탕과자,1.5,33,0.0,0,33
4,2015-01,가나,18.0,코코아와 그 조제품,0.8,28,1223.8,1980,-1952


2. 외국인 투자유치(FDI) 동향

In [101]:
fdi = pd.read_csv('산업별_외국인_투자_유치_실적_201501_202404.csv', encoding='ANSI', header=None)

fdi.head()

,0,1,2,3,4,5,6,7,8,9,...,72,73,74,75,76,77,78,79,80,81
0,산업분류별(1),산업분류별(2),2015.1/4,2015.1/4,2015.2/4,2015.2/4,2015.3/4,2015.3/4,2015.4/4,2015.4/4,...,2023.4/4,2023.4/4,2024.1/4,2024.1/4,2024.2/4,2024.2/4,2024.3/4,2024.3/4,2024.4/4,2024.4/4
1,산업분류별(1),산업분류별(2),신고건수 (개),신고금액 (천불),신고건수 (개),신고금액 (천불),신고건수 (개),신고금액 (천불),신고건수 (개),신고금액 (천불),...,신고건수 (개),신고금액 (천불),신고건수 (개),신고금액 (천불),신고건수 (개),신고금액 (천불),신고건수 (개),신고금액 (천불),신고건수 (개),신고금액 (천불)
2,계,소계,558,3554801,742,5316022,624,4400700,774,7638758,...,933,8771794,847,7054414,922,8281261,882,9848698,977,9385808
3,농·축·수산·광업,소계,3,291,3,494,2,3518,4,1416,...,5,70189,1,77,5,68988,3,293,2,237
4,농·축·수산·광업,농·축·임업,3,291,3,494,1,3433,2,273,...,4,70104,1,77,5,68988,3,293,2,237


# 전처리

- 날짜 컬럼 datetime 변환 & 일별 -> 월별 평균

In [121]:
# 각 데이터 날짜 형식 확인
for name, df in [("ex_rate", ex_rate), ("kr_rate", kr_rate), ("us_rate", us_rate),
                 ("wti", wti), ("dubai", dubai), ("cpi", cpi)]:
    print(f"{name}: {df['날짜'].iloc[0]} / dtype: {df['날짜'].dtype}")

ex_rate: 20150102 / dtype: str
kr_rate: 201501 / dtype: str
us_rate: 201501 / dtype: str
wti: 15년01월02일 / dtype: str
dubai: 15년01월02일 / dtype: str
cpi: 201501 / dtype: str


In [124]:
# 1. 모든 데이터 날짜 컬럼 datetime 변환

# 20150101 형식 -> datetime (ex_rate)
ex_rate["날짜"] = pd.to_datetime(ex_rate["날짜"], format="%Y%m%d")

# 201501 형식 → datetime (kr_rate, us_rate, cpi)
for df in [kr_rate, us_rate, cpi]:
    df["날짜"] = pd.to_datetime(df["날짜"], format="%Y%m")

# 15년01월02일 형식 → datetime (wti, dubai)
for df in [wti, dubai]:
    df["날짜"] = pd.to_datetime(df["날짜"], format="%y년%m월%d일")

# 확인
for name, df in [("ex_rate", ex_rate), ("kr_rate", kr_rate), ("us_rate", us_rate),
                 ("wti", wti), ("dubai", dubai), ("cpi", cpi)]:
    print(f"{name}: {df['날짜'].iloc[0]} / dtype: {df['날짜'].dtype}")

ex_rate: 2015-01-02 00:00:00 / dtype: datetime64[us]
kr_rate: 2015-01-01 00:00:00 / dtype: datetime64[us]
us_rate: 2015-01-01 00:00:00 / dtype: datetime64[us]
wti: 2015-01-02 00:00:00 / dtype: datetime64[us]
dubai: 2015-01-02 00:00:00 / dtype: datetime64[us]
cpi: 2015-01-01 00:00:00 / dtype: datetime64[us]


In [128]:
trade = trade.rename(columns={'기간':'날짜'})

trade["날짜"] = pd.to_datetime(trade["날짜"], format="%Y-%m")

In [129]:
dfs = [ex_rate, kr_rate, us_rate, wti, dubai, cpi, trade]

for df in dfs:
    df["날짜"] = df["날짜"].dt.to_period("M")

# 확인
for name, df in [("ex_rate", ex_rate), ("kr_rate", kr_rate), ("wti", wti)]:
    print(f"{name}: {df['날짜'].iloc[0]} / dtype: {df['날짜'].dtype}")

ex_rate: 2015-01 / dtype: period[M]
kr_rate: 2015-01 / dtype: period[M]
wti: 2015-01 / dtype: period[M]


In [131]:
# 2. 일별 → 월별 평균 (일별 데이터만)
daily_dfs = {
    "ex_rate": (ex_rate, "원/달러환율"),
    "wti":     (wti,     "WTI"),
    "dubai":   (dubai,   "Dubai"),
}

monthly = {}
for name, (df, col) in daily_dfs.items():
    monthly[name] = (df.groupby(df["날짜"])[col]
                       .mean()
                       .reset_index()
                       .rename(columns={"날짜": "날짜"}))

# 결과 확인
for name, df in monthly.items():
    print(f"=== {name} ===")
    print(df.head(3))

=== ex_rate ===
        날짜       원/달러환율
0  2015-01  1088.861905
1  2015-02  1098.400000
2  2015-03  1112.568182
=== wti ===
        날짜        WTI
0  2015-01  47.325500
1  2015-02  50.724737
2  2015-03  47.854091
=== dubai ===
        날짜      Dubai
0  2015-01  45.774762
1  2015-02  55.688333
2  2015-03  54.693636


- FDI 데이터 Wide -> Long 변환

In [102]:
import numpy as np

# 분기·항목 컬럼명 조합
quarters = fdi.iloc[0, 2:].tolist()
items = fdi.iloc[1, 2:].tolist()

filled_q, last = [], ''
for q in quarters:
    if str(q) != 'nan' and q != last:
        last = q
    filled_q.append(last)

col_names = ['산업대분류', '산업소분류'] + [f'{q}_{it}' for q, it in zip(filled_q, items)]
data = fdi.iloc[2:].copy()
data.columns = col_names
data = data.reset_index(drop = True)

In [103]:
# Wide -> Long 변환
fdi_long = data.melt(id_vars=['산업대분류', '산업소분류'], var_name='분기_항목', value_name='값')
fdi_long['연도'] = fdi_long['분기_항목'].str.extract(r'(\d{4})')
fdi_long['분기'] = fdi_long['분기_항목'].str.extract(r'(\d)/4')
fdi_long['항목'] = fdi_long['분기_항목'].str.extract(r'_(.+)')
fdi_long = fdi_long.drop(columns='분기_항목')

# 신고금액만 추출, 소계 행만
fdi = (fdi_long[fdi_long['항목'] == '신고금액 (천불)']
       .drop(columns='항목')
       .rename(columns={'값': '신고금액(천불)'})
       .reset_index(drop=True))

fdi['신고금액(천불)'] = pd.to_numeric(fdi['신고금액(천불)'].replace('-', np.nan))

print(fdi.shape)
fdi.head(10)

(1320, 5)


,산업대분류,산업소분류,신고금액(천불),연도,분기
0,계,소계,3554801.0,2015,1
1,농·축·수산·광업,소계,291.0,2015,1
2,농·축·수산·광업,농·축·임업,291.0,2015,1
3,농·축·수산·광업,어업,0.0,2015,1
4,농·축·수산·광업,광업,0.0,2015,1
5,제조업,소계,385270.0,2015,1
6,제조업,식품,2340.0,2015,1
7,제조업,섬유·직물·의류,95.0,2015,1
8,제조업,제지·목재,1677.0,2015,1
9,제조업,화공,101019.0,2015,1


In [104]:
print(fdi['산업대분류'].unique())
print(fdi['산업소분류'].unique())

<StringArray>
['계', '농·축·수산·광업', '제조업', '서비스업', '전기·가스·수도·환경정화·건설']
Length: 5, dtype: str
<StringArray>
[          '소계',       '농·축·임업',           '어업',           '광업',
           '식품',     '섬유·직물·의류',        '제지·목재',           '화공',
           '의약',     '비금속 광물제품',    '금속·금속가공제품',    '기계장비·의료정밀',
        '전기·전자',        '운송용기계',         '기타제조',     '도·소매(유통)',
       '숙박·음식점',        '운수·창고',         '정보통신',        '금융·보험',
          '부동산',      '사업지원·임대', '연구개발·전문·과학기술',    '여가·스포츠·오락',
     '공공·기타서비스',        '전기·가스',   '수도·하수·환경정화',         '종합건설',
      '전문직별 공사']
Length: 29, dtype: str
